# FFT Pricer — Effect of α and N on Call Prices
This notebook runs `fft_pricer` across multiple damping factors `α` and grid sizes `N`, and plots the results to visualize the effect of adaptive dampening.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from fourier_options.pricing.fft_pricer import fft_pricer
from fourier_options.domain.characteristic_functions import cf_bs
from tests.test_pricer import black_scholes_price, check_convexity

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
params = {
    "S0":    100.0,
    "r":     0.05,
    "T":     1.0,
    "sigma": 0.2
}

ALPHAS = [0.25, 0.75, 1.25, 2.0, 4.0]   # damping factors to test
NS     = [256, 1024, 4096]               # grid sizes to test
ETA    = 0.25

S0   = params["S0"]
mask = lambda K: (K > S0 * 0.4) & (K < S0 * 1.8)

# Analytical benchmark on a fine strike grid
K_bench  = np.linspace(S0 * 0.4, S0 * 1.8, 500)
bs_bench = black_scholes_price(
    S0, K_bench, params["T"], params["r"], params["sigma"], "call"
)

## 1 — Effect of α (fixed N = 4096)

In [ ]:
fig, axes = plt.subplots(len(ALPHAS), 1, figsize=(11, 3.5 * len(ALPHAS)), sharex=True)

for ax, alpha in zip(axes, ALPHAS):
    K, prices      = fft_pricer(cf_bs, params, alpha=alpha, N=4096, eta=ETA)
    K_c            = K[mask(K)]
    prices_c       = prices[mask(K)]
    is_convex      = check_convexity(prices_c, tolerance=1e-6)
    color          = "green" if is_convex else "red"
    status         = "✓ convex" if is_convex else "✗ convexity violated"

    ax.plot(K_bench, bs_bench, "k--", linewidth=1.2, label="BS analytical", zorder=1)
    ax.plot(K_c, prices_c, color="steelblue", linewidth=1.5, label=f"FFT α={alpha}", zorder=2)
    ax.set_title(f"α = {alpha}  —  {status}", color=color, fontsize=11)
    ax.set_ylabel("Call Price")
    ax.legend(fontsize=9)
    ax.grid(True, linestyle=":")

axes[-1].set_xlabel("Strike K")
fig.suptitle("Effect of Damping Factor α on FFT Prices  (N=4096)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 2 — Effect of N (fixed α = 1.25)

In [ ]:
fig, axes = plt.subplots(len(NS), 1, figsize=(11, 3.5 * len(NS)), sharex=True)

for ax, N in zip(axes, NS):
    K, prices      = fft_pricer(cf_bs, params, alpha=1.25, N=N, eta=ETA)
    K_c            = K[mask(K)]
    prices_c       = prices[mask(K)]
    is_convex      = check_convexity(prices_c, tolerance=1e-6)
    color          = "green" if is_convex else "red"
    status         = "✓ convex" if is_convex else "✗ convexity violated"

    ax.plot(K_bench, bs_bench, "k--", linewidth=1.2, label="BS analytical", zorder=1)
    ax.plot(K_c, prices_c, color="darkorange", linewidth=1.5, label=f"FFT N={N}", zorder=2)
    ax.set_title(f"N = {N}  —  {status}", color=color, fontsize=11)
    ax.set_ylabel("Call Price")
    ax.legend(fontsize=9)
    ax.grid(True, linestyle=":")

axes[-1].set_xlabel("Strike K")
fig.suptitle("Effect of Grid Size N on FFT Prices  (α=1.25)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3 — Heatmap: Max Absolute Error vs α and N

In [ ]:
errors    = np.zeros((len(ALPHAS), len(NS)))
convexity = np.zeros((len(ALPHAS), len(NS)), dtype=bool)

for i, alpha in enumerate(ALPHAS):
    for j, N in enumerate(NS):
        K, prices  = fft_pricer(cf_bs, params, alpha=alpha, N=N, eta=ETA)
        K_c        = K[mask(K)]
        prices_c   = prices[mask(K)]
        bs_c       = black_scholes_price(
            S0, K_c, params["T"], params["r"], params["sigma"], "call"
        )
        errors[i, j]    = np.max(np.abs(prices_c - bs_c))
        convexity[i, j] = check_convexity(prices_c, tolerance=1e-6)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Max error heatmap
im0 = axes[0].imshow(errors, aspect="auto", cmap="YlOrRd")
axes[0].set_xticks(range(len(NS)));     axes[0].set_xticklabels(NS)
axes[0].set_yticks(range(len(ALPHAS))); axes[0].set_yticklabels(ALPHAS)
axes[0].set_xlabel("N"); axes[0].set_ylabel("α")
axes[0].set_title("Max Absolute Error vs Analytical BS")
plt.colorbar(im0, ax=axes[0])
for i in range(len(ALPHAS)):
    for j in range(len(NS)):
        axes[0].text(j, i, f"{errors[i,j]:.4f}", ha="center", va="center", fontsize=9)

# Convexity heatmap
im1 = axes[1].imshow(convexity.astype(float), aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
axes[1].set_xticks(range(len(NS)));     axes[1].set_xticklabels(NS)
axes[1].set_yticks(range(len(ALPHAS))); axes[1].set_yticklabels(ALPHAS)
axes[1].set_xlabel("N"); axes[1].set_ylabel("α")
axes[1].set_title("Convexity Satisfied  (green=✓, red=✗)")
for i in range(len(ALPHAS)):
    for j in range(len(NS)):
        axes[1].text(j, i, "✓" if convexity[i,j] else "✗", ha="center", va="center", fontsize=12)

fig.suptitle("α × N Grid — Error and Convexity Summary", fontsize=13)
plt.tight_layout()
plt.show()

## 4 — Adaptive α selection

In [ ]:
K_auto, prices_auto = fft_pricer(cf_bs, params)  # alpha=None → adaptive
K_c                 = K_auto[mask(K_auto)]
prices_c            = prices_auto[mask(K_auto)]
is_convex           = check_convexity(prices_c, tolerance=1e-6)

plt.figure(figsize=(10, 4))
plt.plot(K_bench,  bs_bench,  "k--", linewidth=1.2, label="BS analytical")
plt.plot(K_c,      prices_c,  color="mediumseagreen", linewidth=2, label="FFT adaptive α")
plt.title(
    f"Adaptive α selection  —  {'✓ convex' if is_convex else '✗ convexity violated'}",
    color="green" if is_convex else "red"
)
plt.xlabel("Strike K")
plt.ylabel("Call Price")
plt.legend()
plt.grid(True, linestyle=":")
plt.tight_layout()
plt.show()